# Sen1Floods11 — Flood Segmentation with SSL4EO-SAR Pretrained ViT
### Binary Segmentation: Flood vs Non-Flood from Sentinel-1 SAR

**Goal:** Replace the ImageNet-pretrained MiT-B2 encoder with a **SAR-pretrained ViT-B/16** from SSL4EO-S12.
The SSL4EO-S12 weights were pretrained via Masked Autoencoding (MAE) on ~250k Sentinel-1 SAR
patches globally. Since these were pretrained on the *same modality* (2-channel VV+VH SAR),
no patch-embedding re-init is needed — every weight transfers cleanly.

**Architecture:**
- **Encoder:** ViT-B/16 (12 transformer blocks, 768 dim, 86M params) — SSL4EO-SAR weights
- **Decoder:** SETR-PUP style progressive upsampling head
- **Input:** S1 SAR (VV, VH) — 2 bands, 512×512 chips
- **Output:** Binary flood mask (0 = non-flood, 1 = flood)
- **Loss:** Dice + BCE with invalid-pixel masking

**Reference weights:** [wangyi111/SSL4EO-S12 → B2_vitb16_mae_ep99.pth](https://huggingface.co/wangyi111/SSL4EO-S12/resolve/main/B2_vitb16_mae_ep99.pth)

---
**Sections:**
1. Download dataset & SSL4EO-SAR weights
2. Imports & Config
3. Dataset & DataLoaders
4. ViT-B/16 + SETR-PUP decoder
5. Loss
6. Training loop
7. Evaluation (Test + Bolivia held-out)
8. TTA inference
9. Save model

## 0. Setup — Download Sen1Floods11 + SSL4EO-SAR Weights

In [ ]:
# ── Step 1: Authenticate with Google Cloud ─────────────────────────────────
from google.colab import auth
auth.authenticate_user()
print('Authenticated ✓')

# ── Step 2: Install dependencies ───────────────────────────────────────────
!pip install -q rasterio timm einops
import rasterio, timm
print(f'rasterio  ✓')
print(f'timm {timm.__version__} ✓')

In [ ]:
# ── Step 3: Download Sen1Floods11 ──────────────────────────────────────────
import os
DATA_DIR   = '/content/sen1floods11'
S1_PATH    = os.path.join(DATA_DIR, 'S1')
LABEL_PATH = os.path.join(DATA_DIR, 'Labels')
SPLIT_PATH = os.path.join(DATA_DIR, 'splits')
for p in (S1_PATH, LABEL_PATH, SPLIT_PATH):
    os.makedirs(p, exist_ok=True)

if len(os.listdir(S1_PATH)) > 400:
    print('Dataset already downloaded, skipping.')
else:
    print('[1/3] Downloading split files...')
    !gsutil -q cp gs://sen1floods11/v1.1/splits/flood_handlabeled/flood_train_data.csv {SPLIT_PATH}/
    !gsutil -q cp gs://sen1floods11/v1.1/splits/flood_handlabeled/flood_valid_data.csv {SPLIT_PATH}/
    !gsutil -q cp gs://sen1floods11/v1.1/splits/flood_handlabeled/flood_test_data.csv  {SPLIT_PATH}/
    !gsutil -q cp gs://sen1floods11/v1.1/splits/flood_handlabeled/flood_bolivia_data.csv {SPLIT_PATH}/
    print('[2/3] Downloading S1 SAR images (446 chips)...')
    !gsutil -m -q rsync -r gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand     {S1_PATH}
    print('[3/3] Downloading expert labels (446 chips)...')
    !gsutil -m -q rsync -r gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand  {LABEL_PATH}
    print('Download complete ✓')

print(f'  S1 chips:    {len([f for f in os.listdir(S1_PATH) if f.endswith(".tif")])}')
print(f'  Label chips: {len([f for f in os.listdir(LABEL_PATH) if f.endswith(".tif")])}')

In [ ]:
# ── Step 4: Download SSL4EO-S12 SAR-pretrained ViT-B/16 ────────────────────
# This is a 2-channel ViT-B/16 pretrained via MAE on Sentinel-1 SAR.
# Same modality as our task → no patch embedding re-init required.
from pathlib import Path
import urllib.request

WEIGHTS_DIR = Path('/content/ssl4eo_weights')
WEIGHTS_DIR.mkdir(exist_ok=True)
WEIGHTS_PATH = WEIGHTS_DIR / 'B2_vitb16_mae_ep99.pth'

if not WEIGHTS_PATH.exists():
    print('Downloading SSL4EO-SAR ViT-B/16 weights (~330 MB)...')
    url = 'https://huggingface.co/wangyi111/SSL4EO-S12/resolve/main/B2_vitb16_mae_ep99.pth'
    urllib.request.urlretrieve(url, WEIGHTS_PATH)
    print(f'Saved to {WEIGHTS_PATH}')
else:
    print(f'Weights already present: {WEIGHTS_PATH}')
print(f'  size: {WEIGHTS_PATH.stat().st_size / 1e6:.1f} MB')

In [ ]:
# ── Step 5: Mount Drive for checkpointing ──────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
GDRIVE_CKPT_DIR = Path('/content/drive/MyDrive/sen1floods11_checkpoints')
GDRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = GDRIVE_CKPT_DIR / 'ssl4eo_vitb_flood_checkpoint.pt'
print(f'Checkpoint path: {CKPT_PATH}')

## 1. Imports & Configuration

In [ ]:
import os, random, copy, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from collections import OrderedDict
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import rasterio
import timm
from tqdm.auto import tqdm

print(f'PyTorch: {torch.__version__}')
print(f'timm:    {timm.__version__}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Using device: {DEVICE}')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
S1_DIR    = Path('/content/sen1floods11/S1')
LABEL_DIR = Path('/content/sen1floods11/Labels')
SPLIT_DIR = Path('/content/sen1floods11/splits')

# ── Normalization (same constants as the SegFormer notebook for fair comparison) ──
VV_MEAN, VV_STD = -10.41, 4.14
VH_MEAN, VH_STD = -17.14, 4.68

# ── Hyperparameters ────────────────────────────────────────────────────────
# ViT-B/16 has ~86M params → a bit heavier than MiT-B2 (27M).
# Lower batch size + slightly higher LR with cosine schedule + warmup tend to work
# well for MAE-pretrained ViTs on dense prediction tasks.
IMG_SIZE      = 512
PATCH_SIZE    = 16
BATCH_SIZE    = 8       # ViT-B at 512×512 is memory-heavy; reduce if OOM on Colab
EPOCHS        = 100
LR_BACKBONE   = 1e-5    # SSL-pretrained backbone — needs gentle fine-tuning
LR_DECODER    = 1e-4    # decoder is randomly init'd → higher LR
WEIGHT_DECAY  = 0.05    # standard for ViTs
WARMUP_EPOCHS = 5
DICE_WEIGHT   = 0.5
BCE_WEIGHT    = 0.5
PATIENCE      = 15
NUM_WORKERS   = 8

print('Config set ✓')

## 2. Dataset & DataLoaders
Same dataset class as the SegFormer notebook — identical splits and augmentation
for an apples-to-apples comparison.

In [ ]:
class Sen1Floods11Dataset(Dataset):
    def __init__(self, csv_path, s1_dir, label_dir, augment=False):
        self.s1_dir, self.label_dir, self.augment = Path(s1_dir), Path(label_dir), augment
        self.pairs = []
        with open(csv_path) as f:
            for line in f:
                line = line.strip()
                if not line: continue
                s1_name, label_name = line.split(',')
                self.pairs.append((s1_name.strip(), label_name.strip()))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        s1_name, label_name = self.pairs[idx]
        with rasterio.open(self.s1_dir / s1_name) as src:
            s1 = src.read().astype(np.float32)
        with rasterio.open(self.label_dir / label_name) as src:
            label = src.read(1).astype(np.float32)

        s1[0] = (s1[0] - VV_MEAN) / VV_STD
        s1[1] = (s1[1] - VH_MEAN) / VH_STD
        s1 = np.nan_to_num(s1, nan=0.0)

        valid_mask = (label != -1).astype(np.float32)
        label = np.clip(label, 0, 1)

        if self.augment:
            s1, label, valid_mask = self._augment(s1, label, valid_mask)

        return {
            'image':      torch.from_numpy(s1.copy()),
            'label':      torch.from_numpy(label.copy()).unsqueeze(0),
            'valid_mask': torch.from_numpy(valid_mask.copy()).unsqueeze(0),
            'chip_id':    s1_name.replace('_S1Hand.tif', ''),
        }

    def _augment(self, image, label, valid_mask):
        if random.random() > 0.5:
            image = np.flip(image, axis=2); label = np.flip(label, axis=1); valid_mask = np.flip(valid_mask, axis=1)
        if random.random() > 0.5:
            image = np.flip(image, axis=1); label = np.flip(label, axis=0); valid_mask = np.flip(valid_mask, axis=0)
        k = random.randint(0, 3)
        if k > 0:
            image = np.rot90(image, k, axes=(1, 2))
            label = np.rot90(label, k, axes=(0, 1))
            valid_mask = np.rot90(valid_mask, k, axes=(0, 1))
        # ── SAR speckle noise augmentation (multiplicative) ───────────────
        # Helps the model become robust to SAR speckle variance — a real-world signal
        # property that doesn't appear in ImageNet/optical pretraining.
        if random.random() > 0.5:
            speckle = np.random.normal(0, 0.05, image.shape).astype(np.float32)
            image = image * (1 + speckle)
        return image, label, valid_mask

train_ds   = Sen1Floods11Dataset(SPLIT_DIR/'flood_train_data.csv',   S1_DIR, LABEL_DIR, augment=True)
val_ds     = Sen1Floods11Dataset(SPLIT_DIR/'flood_valid_data.csv',   S1_DIR, LABEL_DIR, augment=False)
test_ds    = Sen1Floods11Dataset(SPLIT_DIR/'flood_test_data.csv',    S1_DIR, LABEL_DIR, augment=False)
bolivia_ds = Sen1Floods11Dataset(SPLIT_DIR/'flood_bolivia_data.csv', S1_DIR, LABEL_DIR, augment=False)

loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                     persistent_workers=NUM_WORKERS>0, prefetch_factor=2 if NUM_WORKERS>0 else None,
                     pin_memory=True)
train_loader   = DataLoader(train_ds,   shuffle=True,  **loader_kwargs)
val_loader     = DataLoader(val_ds,     shuffle=False, **loader_kwargs)
test_loader    = DataLoader(test_ds,    shuffle=False, **loader_kwargs)
bolivia_loader = DataLoader(bolivia_ds, shuffle=False, **loader_kwargs)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)} | Bolivia: {len(bolivia_ds)}')
batch = next(iter(train_loader))
print(f'Image batch: {tuple(batch["image"].shape)} | Label batch: {tuple(batch["label"].shape)}')

## 3. ViT-B/16 (SSL4EO-SAR) + SETR-PUP Decoder

**Encoder:** Standard ViT-B/16 from `timm` with `in_chans=2`. We load SSL4EO-S12's MAE weights
(SAR-pretrained, 2-channel) directly — no patch-embedding re-initialization needed.

**Decoder (SETR-PUP):** Progressive 4× upsampling on the patch grid:
- ViT outputs `(B, 1024, 768)` for a 512×512 input with 16×16 patches → reshape to `(B, 768, 32, 32)`
- Four conv-BN-ReLU + 2× upsample blocks bring it to the full 512×512 resolution
- A 1×1 conv produces the final logit map

This is the original SETR design (Zheng et al., CVPR 2021) — simple, well-validated
for ViT-based dense prediction, and avoids the complexity of UperNet's multi-scale
feature pyramid (which requires picking and reshaping intermediate transformer layers).

In [ ]:
def build_vit_encoder(weights_path, img_size=512, patch_size=16, in_chans=2):
    """Build a 2-channel ViT-B/16 and load SSL4EO-SAR pretrained weights."""
    # timm's ViT-B/16 — note: img_size matters for positional embeddings.
    # SSL4EO-S12 was pretrained at 224 → we'll need to interpolate the pos embeds to 512.
    model = timm.create_model(
        'vit_base_patch16_224',
        pretrained=False,
        in_chans=in_chans,
        num_classes=0,           # remove classification head
        img_size=img_size,       # timm supports adaptive pos-embed interpolation here
    )

    # ── Load SSL4EO-SAR weights ───────────────────────────────────────────
    import argparse
    torch.serialization.add_safe_globals([argparse.Namespace])
    ckpt = torch.load(weights_path, map_location='cpu', weights_only=True)
    state_dict = ckpt.get('model', ckpt.get('state_dict', ckpt))

    # The MAE checkpoint may have an 'encoder.' or 'module.' prefix — strip it.
    cleaned = OrderedDict()
    for k, v in state_dict.items():
        nk = k
        for prefix in ('module.', 'encoder.', 'backbone.'):
            if nk.startswith(prefix):
                nk = nk[len(prefix):]
        # Drop MAE-only keys (decoder, mask token) — we only need the encoder.
        if nk.startswith('decoder') or nk in ('mask_token',):
            continue
        cleaned[nk] = v

    # ── Interpolate positional embeddings if pretrained at 224, fine-tuning at 512 ──
    if 'pos_embed' in cleaned:
        pretrained_pe = cleaned['pos_embed']        # (1, N_pre+1, dim)
        target_pe     = model.state_dict()['pos_embed']
        if pretrained_pe.shape != target_pe.shape:
            print(f'Interpolating pos_embed: {tuple(pretrained_pe.shape)} → {tuple(target_pe.shape)}')
            cls_pe, grid_pe = pretrained_pe[:, :1], pretrained_pe[:, 1:]
            n_pre   = grid_pe.shape[1]
            n_post  = target_pe.shape[1] - 1
            side_pre, side_post = int(n_pre ** 0.5), int(n_post ** 0.5)
            dim = grid_pe.shape[-1]
            grid_pe = grid_pe.reshape(1, side_pre, side_pre, dim).permute(0, 3, 1, 2)
            grid_pe = F.interpolate(grid_pe, size=(side_post, side_post), mode='bicubic', align_corners=False)
            grid_pe = grid_pe.permute(0, 2, 3, 1).reshape(1, n_post, dim)
            cleaned['pos_embed'] = torch.cat([cls_pe, grid_pe], dim=1)

    missing, unexpected = model.load_state_dict(cleaned, strict=False)
    print(f'Loaded SSL4EO-SAR ViT-B/16 ✓')
    print(f'  Missing keys:    {len(missing)}    (e.g. {missing[:3] if missing else []})')
    print(f'  Unexpected keys: {len(unexpected)} (e.g. {unexpected[:3] if unexpected else []})')
    return model


class SETRPUPDecoder(nn.Module):
    """SETR Progressive Upsampling decoder.

    Takes ViT patch-grid features and upsamples 4× via stacked conv blocks
    until the original H×W resolution is recovered.
    """

    def __init__(self, in_dim=768, num_classes=1, hidden=256):
        super().__init__()
        self.up1 = self._block(in_dim, hidden)
        self.up2 = self._block(hidden, hidden)
        self.up3 = self._block(hidden, hidden)
        self.up4 = self._block(hidden, hidden)
        self.head = nn.Conv2d(hidden, num_classes, kernel_size=1)

    @staticmethod
    def _block(c_in, c_out):
        return nn.Sequential(
            nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
        )

    def forward(self, x):
        # x: (B, dim, H/16, W/16)  →  upsample 16× to (B, num_classes, H, W)
        x = self.up1(x); x = self.up2(x); x = self.up3(x); x = self.up4(x)
        return self.head(x)


class SSL4EOSegmenter(nn.Module):
    """SSL4EO-SAR ViT-B/16 encoder + SETR-PUP decoder for binary flood segmentation."""

    def __init__(self, weights_path, img_size=512, patch_size=16, in_chans=2, num_classes=1):
        super().__init__()
        self.encoder = build_vit_encoder(weights_path, img_size=img_size,
                                         patch_size=patch_size, in_chans=in_chans)
        self.embed_dim   = self.encoder.embed_dim          # 768 for ViT-B
        self.patch_size  = patch_size
        self.img_size    = img_size
        self.grid_size   = img_size // patch_size          # 32
        self.decoder = SETRPUPDecoder(in_dim=self.embed_dim, num_classes=num_classes)

    def forward(self, x):
        B, _, H, W = x.shape
        # ── Run through the ViT and return all patch tokens (drop CLS) ───────
        feats = self.encoder.forward_features(x)            # (B, 1+N, dim)
        if feats.dim() == 3:
            feats = feats[:, 1:, :]                         # drop CLS → (B, N, dim)
        # Reshape (B, N, dim) → (B, dim, gh, gw)
        gh = gw = self.grid_size
        feats = feats.transpose(1, 2).reshape(B, self.embed_dim, gh, gw)
        logits = self.decoder(feats)                        # (B, 1, H, W)
        # In case decoder output isn't exactly H×W (rare), interpolate.
        if logits.shape[-2:] != (H, W):
            logits = F.interpolate(logits, size=(H, W), mode='bilinear', align_corners=False)
        return logits


model = SSL4EOSegmenter(
    weights_path=str(WEIGHTS_PATH),
    img_size=IMG_SIZE,
    patch_size=PATCH_SIZE,
    in_chans=2,
    num_classes=1,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
n_enc    = sum(p.numel() for p in model.encoder.parameters())
n_dec    = sum(p.numel() for p in model.decoder.parameters())
print(f'\nModel built ✓')
print(f'  Total params:   {n_params/1e6:.1f}M')
print(f'  Encoder (ViT):  {n_enc/1e6:.1f}M  [SSL4EO-SAR pretrained]')
print(f'  Decoder (PUP):  {n_dec/1e6:.1f}M  [random init]')

In [ ]:
# ── Sanity check: forward pass on a real batch ────────────────────────────
model.eval()
with torch.no_grad():
    out = model(batch['image'].to(DEVICE))
print(f'Forward pass ✓   input {tuple(batch["image"].shape)} → output {tuple(out.shape)}')
assert out.shape == (BATCH_SIZE, 1, IMG_SIZE, IMG_SIZE), 'Output shape mismatch'

## 4. Loss Function — Same Dice + BCE as SegFormer notebook

In [ ]:
class CombinedLoss(nn.Module):
    """Dice + BCE with masking for invalid (-1) pixels."""
    def __init__(self, dice_weight=0.5, bce_weight=0.5, smooth=1.0):
        super().__init__()
        self.dw, self.bw, self.smooth = dice_weight, bce_weight, smooth

    def forward(self, logits, targets, valid_mask):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        bce_masked = (bce * valid_mask).sum() / (valid_mask.sum() + 1e-8)
        probs  = torch.sigmoid(logits) * valid_mask
        target = targets * valid_mask
        intersection = (probs * target).sum()
        dice = (2.0 * intersection + self.smooth) / (probs.sum() + target.sum() + self.smooth)
        return self.dw * (1 - dice) + self.bw * bce_masked

criterion = CombinedLoss(DICE_WEIGHT, BCE_WEIGHT)
print('Loss: Dice + BCE (masked) ✓')

## 5. Training Loop

Two key changes from the SegFormer training:
1. **Two-LR optimizer** — backbone gets 1e-5 (gentle SSL fine-tuning), decoder gets 1e-4 (random init)
2. **Cosine schedule + warmup** — standard recipe for ViT fine-tuning

In [ ]:
def compute_iou(logits, targets, valid_mask, threshold=0.5):
    preds = (torch.sigmoid(logits) > threshold).float() * valid_mask
    targs = targets * valid_mask
    inter = (preds * targs).sum()
    union = preds.sum() + targs.sum() - inter
    return (inter / (union + 1e-8)).item()


def train_one_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    running_loss, running_iou = 0.0, 0.0
    for batch in tqdm(loader, desc='  Train', leave=False):
        images     = batch['image'].to(device)
        labels     = batch['label'].to(device)
        valid_mask = batch['valid_mask'].to(device)
        optimizer.zero_grad()
        with autocast():
            logits = model(images)
            loss   = criterion(logits, labels, valid_mask)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
        running_iou  += compute_iou(logits.float(), labels, valid_mask)
    n = len(loader)
    return running_loss / n, running_iou / n


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    running_loss, running_iou = 0.0, 0.0
    for batch in tqdm(loader, desc='  Val', leave=False):
        images     = batch['image'].to(device)
        labels     = batch['label'].to(device)
        valid_mask = batch['valid_mask'].to(device)
        with autocast():
            logits = model(images)
            loss   = criterion(logits, labels, valid_mask)
        running_loss += loss.item()
        running_iou  += compute_iou(logits.float(), labels, valid_mask)
    n = len(loader)
    return running_loss / n, running_iou / n

In [ ]:
# ── Two-group optimizer: backbone (low LR) vs decoder (high LR) ───────────
# Standard practice when fine-tuning a pretrained encoder with a fresh head:
# - The encoder weights are already meaningful → take small steps
# - The decoder is random → needs larger steps to converge in reasonable time
param_groups = [
    {'params': model.encoder.parameters(), 'lr': LR_BACKBONE, 'weight_decay': WEIGHT_DECAY},
    {'params': model.decoder.parameters(), 'lr': LR_DECODER,  'weight_decay': WEIGHT_DECAY},
]
optimizer = torch.optim.AdamW(param_groups)

# Cosine schedule with linear warmup — standard ViT fine-tuning recipe
from torch.optim.lr_scheduler import LambdaLR
import math
def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + math.cos(math.pi * progress))
scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)
scaler = GradScaler()

# ── Resume if checkpoint exists ───────────────────────────────────────────
history = {'train_loss': [], 'val_loss': [], 'train_iou': [], 'val_iou': [], 'lr_enc': [], 'lr_dec': []}
best_val_iou, best_epoch, best_weights, no_improve, start_epoch = 0.0, 0, None, 0, 1

if CKPT_PATH.exists():
    print(f'Resuming from {CKPT_PATH}...')
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    scaler.load_state_dict(ckpt['scaler_state_dict'])
    history       = ckpt['history']
    best_val_iou  = ckpt['best_val_iou']
    best_epoch    = ckpt['best_epoch']
    best_weights  = ckpt['best_weights']
    no_improve    = ckpt['no_improve']
    start_epoch   = ckpt['epoch'] + 1
    print(f'  ► Resumed from epoch {ckpt["epoch"]}, best val IoU: {best_val_iou:.4f}')
else:
    print('Training from scratch.')

# ── Train ─────────────────────────────────────────────────────────────────
print(f'\nTraining for up to {EPOCHS} epochs (start: {start_epoch})')
print(f"{'Epoch':>5} {'TrLoss':>9} {'VlLoss':>9} {'TrIoU':>9} {'VlIoU':>9} {'LRenc':>10} {'LRdec':>10}")
print('-' * 70)

for epoch in range(start_epoch, EPOCHS + 1):
    train_loss, train_iou = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE, scaler)
    val_loss,   val_iou   = validate(model, val_loader, criterion, DEVICE)

    lr_enc = optimizer.param_groups[0]['lr']
    lr_dec = optimizer.param_groups[1]['lr']
    history['train_loss'].append(train_loss); history['val_loss'].append(val_loss)
    history['train_iou'].append(train_iou);   history['val_iou'].append(val_iou)
    history['lr_enc'].append(lr_enc);         history['lr_dec'].append(lr_dec)

    if val_iou > best_val_iou:
        best_val_iou, best_epoch = val_iou, epoch
        best_weights = copy.deepcopy(model.state_dict())
        no_improve, marker = 0, ' ★'
    else:
        no_improve += 1; marker = ''

    print(f'{epoch:>5} {train_loss:>9.4f} {val_loss:>9.4f} {train_iou:>9.4f} {val_iou:>9.4f} {lr_enc:>10.2e} {lr_dec:>10.2e}{marker}')
    scheduler.step()

    torch.save({
        'epoch': epoch,
        'model_state_dict':     model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict':    scaler.state_dict(),
        'best_weights':         best_weights,
        'best_val_iou':         best_val_iou,
        'best_epoch':           best_epoch,
        'no_improve':           no_improve,
        'history':              history,
    }, CKPT_PATH)

    if no_improve >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)')
        break

model.load_state_dict(best_weights)
print(f'\nDone. Best val IoU: {best_val_iou:.4f} at epoch {best_epoch}')

In [ ]:
# ── Training curves ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history['train_loss']) + 1)
axes[0].plot(epochs_range, history['train_loss'], label='Train', color='#2196F3')
axes[0].plot(epochs_range, history['val_loss'],   label='Val',   color='#FF9800')
axes[0].axvline(best_epoch, color='red', linestyle='--', alpha=0.5, label=f'Best ({best_epoch})')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_title('Loss')
axes[1].plot(epochs_range, history['train_iou'], label='Train', color='#2196F3')
axes[1].plot(epochs_range, history['val_iou'],   label='Val',   color='#FF9800')
axes[1].axvline(best_epoch, color='red', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('IoU'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_title('IoU (flood class)')
plt.tight_layout(); plt.show()

## 6. Evaluation — Test + Bolivia Held-Out (with Threshold Tuning)

In [ ]:
@torch.no_grad()
def evaluate_full(model, loader, device, threshold=0.5, tta=False):
    """Aggregate + per-chip metrics. Optionally apply 8-fold flip/rotate TTA."""
    model.eval()
    tp = fp = fn = tn = 0
    chip_ious, chip_ids = [], []
    for batch in tqdm(loader, desc='  Eval', leave=False):
        images     = batch['image'].to(device)
        labels     = batch['label'].to(device)
        valid_mask = batch['valid_mask'].to(device)

        if tta:
            probs_acc = torch.zeros_like(labels)
            n_tta = 0
            for k in range(4):                          # 4 rotations
                for flip in (False, True):              # × 2 flips
                    aug = torch.rot90(images, k, dims=(2, 3))
                    if flip:
                        aug = torch.flip(aug, dims=(3,))
                    p = torch.sigmoid(model(aug))
                    if flip:
                        p = torch.flip(p, dims=(3,))
                    p = torch.rot90(p, -k, dims=(2, 3))
                    probs_acc += p
                    n_tta += 1
            probs = probs_acc / n_tta
        else:
            probs = torch.sigmoid(model(images))

        preds = (probs > threshold).float()
        for i in range(images.size(0)):
            v = valid_mask[i].bool()
            p = preds[i, 0][v[0]]; t = labels[i, 0][v[0]]
            ctp = ((p == 1) & (t == 1)).sum().item()
            cfp = ((p == 1) & (t == 0)).sum().item()
            cfn = ((p == 0) & (t == 1)).sum().item()
            ctn = ((p == 0) & (t == 0)).sum().item()
            tp += ctp; fp += cfp; fn += cfn; tn += ctn
            chip_ious.append(ctp / (ctp + cfp + cfn + 1e-8))
            chip_ids.append(batch['chip_id'][i])

    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)
    iou       = tp / (tp + fp + fn + 1e-8)
    accuracy  = (tp + tn) / (tp + tn + fp + fn + 1e-8)
    return {'IoU': iou, 'F1': f1, 'Precision': precision, 'Recall': recall, 'Accuracy': accuracy}, chip_ious, chip_ids


# ── Threshold tuning on validation set ───────────────────────────────────
print('Tuning threshold on validation set...')
best_thresh, best_thresh_iou = 0.5, 0.0
for t in np.arange(0.30, 0.66, 0.025):
    m, _, _ = evaluate_full(model, val_loader, DEVICE, threshold=float(t), tta=False)
    print(f'  threshold {t:.3f}: val IoU = {m["IoU"]:.4f}')
    if m['IoU'] > best_thresh_iou:
        best_thresh_iou, best_thresh = m['IoU'], float(t)
print(f'\nBest threshold: {best_thresh:.3f}  (val IoU: {best_thresh_iou:.4f})')

In [ ]:
# ── Final evaluation: tuned threshold + TTA ──────────────────────────────
print('=== TEST SET (single forward) ===')
test_metrics_plain, test_chip_ious, test_chip_ids = evaluate_full(
    model, test_loader, DEVICE, threshold=best_thresh, tta=False)
for k, v in test_metrics_plain.items(): print(f'  {k:<10}: {v:.4f}')

print('\n=== TEST SET (with TTA) ===')
test_metrics_tta, _, _ = evaluate_full(
    model, test_loader, DEVICE, threshold=best_thresh, tta=True)
for k, v in test_metrics_tta.items(): print(f'  {k:<10}: {v:.4f}')

print('\n=== BOLIVIA HELD-OUT (with TTA) ===')
bol_metrics, bol_chip_ious, bol_chip_ids = evaluate_full(
    model, bolivia_loader, DEVICE, threshold=best_thresh, tta=True)
for k, v in bol_metrics.items(): print(f'  {k:<10}: {v:.4f}')

print('\n=== Summary vs SegFormer baseline ===')
print(f"  SegFormer MiT-B2 (ImageNet pretrained): Test IoU = 0.6680")
print(f"  SSL4EO ViT-B/16 (SAR pretrained, +TTA):  Test IoU = {test_metrics_tta['IoU']:.4f}")
print(f"  Δ = {test_metrics_tta['IoU'] - 0.6680:+.4f}")

## 7. Save final model

In [ ]:
save_path = GDRIVE_CKPT_DIR / 'ssl4eo_vitb_flood_best.pt'
torch.save({
    'model_state_dict': best_weights,
    'epoch':            best_epoch,
    'val_iou':          best_val_iou,
    'best_threshold':   best_thresh,
    'config': {
        'model':         'ssl4eo-sar-vitb16-setrpup',
        'pretrained':    'wangyi111/SSL4EO-S12 B2_vitb16_mae_ep99',
        'img_size':      IMG_SIZE,
        'patch_size':    PATCH_SIZE,
        'vv_mean':       VV_MEAN, 'vv_std': VV_STD,
        'vh_mean':       VH_MEAN, 'vh_std': VH_STD,
        'lr_backbone':   LR_BACKBONE,
        'lr_decoder':    LR_DECODER,
    },
    'test_metrics_plain': test_metrics_plain,
    'test_metrics_tta':   test_metrics_tta,
    'bolivia_metrics':    bol_metrics,
    'history':            history,
}, save_path)
print(f'Saved → {save_path}')
print(f'  Best val IoU: {best_val_iou:.4f}')
print(f'  Test IoU (TTA, threshold={best_thresh:.2f}): {test_metrics_tta["IoU"]:.4f}')
print(f'  Bolivia IoU (TTA): {bol_metrics["IoU"]:.4f}')